In [3]:
import pandas as pd
import torch

# run on GPU if available
global device
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)

In [12]:

# loading the prepared dataset
x = pd.read_csv("data/x_data.csv").to_numpy()
y = pd.read_csv("data/y_data.csv").to_numpy()


In [ ]:
from torch import Tensor

def construct_sub_matrix(array):
    # fill in the lower triangle of the matrix
    matrix = torch.zeros((3, 3), device=device)
    matrix[2, 2] = array[2]  # 33
    matrix[2, 1] = array[3]  # 23
    matrix[2, 0] = array[4]  # 13
    matrix[1, 0] = array[5]  # 12
    matrix[0, 0] = array[0]  # 11
    matrix[1, 1] = array[1]  # 22
    # mirror lower triangle to upper triangle
    matrix: Tensor = matrix + matrix.T - torch.diag(matrix.diagonal())

    return matrix

def construct_full_matrix(array):
    A: Tensor = construct_sub_matrix(array=array[0:6])
    B: Tensor = construct_sub_matrix(array=array[6:12])
    D: Tensor = construct_sub_matrix(array=array[12:18])
    # construct the full 6x6 matrix
    matrix: Tensor = torch.zeros((6, 6), device=device)
    matrix[0:3, 0:3] = A
    matrix[0:3, 3:6] = B
    matrix[3:6, 0:3] = B
    matrix[3:6, 3:6] = D

    return matrix

# sanity check
print("x:\n", x[0])
print("y:\n", y[0])
matrix = construct_full_matrix(torch.tensor(y[0], device=device))
print("A:\n", matrix[0:3, 0:3])
print("B:\n", matrix[0:3, 3:6])
print("D:\n", matrix[3:6, 3:6])


x:
 [ 0.22905405 -0.01067568  0.          0.          0.          0.        ]
y:
 [ 3.92922916e+01  3.47253911e+02  8.59728303e+00  1.97394770e-02
  1.82106330e-02  2.09769623e+01  4.20000000e-05 -2.10000000e-05
 -4.14000000e-06  2.02000000e-05  2.58000000e-05  6.79000000e-05
  1.26507750e+03  4.57065988e+03  4.11240461e+02  5.05000000e-05
  9.61000000e-05  7.23216636e+02]
A:
 tensor([[3.9292e+01, 2.0977e+01, 1.8211e-02],
        [2.0977e+01, 3.4725e+02, 1.9739e-02],
        [1.8211e-02, 1.9739e-02, 8.5973e+00]], device='cuda:0')
B:
 tensor([[ 4.2000e-05,  6.7900e-05,  2.5800e-05],
        [ 6.7900e-05, -2.1000e-05,  2.0200e-05],
        [ 2.5800e-05,  2.0200e-05, -4.1400e-06]], device='cuda:0')
D:
 tensor([[1.2651e+03, 7.2322e+02, 9.6100e-05],
        [7.2322e+02, 4.5707e+03, 5.0500e-05],
        [9.6100e-05, 5.0500e-05, 4.1124e+02]], device='cuda:0')
